# Agent: weather

Develop and test **`agentic_scd.agents.weather.weather_node`** in isolation.

This is the Weather Risk Monitoring agent (README agent #3). It runs between `news` and `classify`, turning the 7-day Open-Meteo forecast carried by each `WEATHER` signal into a structured hub-level risk assessment.

## What this agent does

```mermaid
flowchart LR
    U["new_signals<br/>DisruptionSignal[] (WEATHER only)"]:::faded --> A1
    subgraph A["weather_node (per WEATHER signal)"]
        A1["read raw_payload.response<br/>(Open-Meteo daily block)"] --> A2["parse_daily_series -> 7 days"]
        A2 --> A3["score_hub_risk (peak + persistence)"]
        A3 --> A4["operations_at_risk + summary"]
    end
    A4 --> D["weather_risks<br/>WeatherRiskAssessment[]"]
    D --> DOWN["downstream: classify (severity boost) · impact (hub->lane)"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```

**State contract**

- **Reads:** `new_signals` (only records with `source_type == "WEATHER"`; non-weather signals pass through untouched)
- **Writes:** `weather_risks` (list of `WeatherRiskAssessment`: aggregate severity, port disruption risk, affected operations, per-day forecast)
- **Fallback / degradation:** offline by design — it reads the structured payload the connector already persisted and never re-fetches; a signal missing a usable `response` is skipped.

## Build weather signals from the packaged Open-Meteo fallback

The `sample_state` helper emits synthetic (non-weather) signals, so here we build real `WEATHER` signals from the committed 7-day fallback snapshot the connector ships with — the same shape ingestion persists.

In [ ]:
from agentic_scd.ingestion.connectors.open_meteo import OpenMeteoConnector
from agentic_scd.ingestion.paths import FALLBACK_DIR
from agentic_scd.ingestion.normalize import normalize

connector = OpenMeteoConnector("open_meteo", 0.9, hubs=[], fallback_path=FALLBACK_DIR / "open_meteo_hubs.json")
state = {"new_signals": [normalize(item, connector) for item in connector.fallback()]}
print("Input WEATHER signals:")
for s in state["new_signals"]:
    print(" -", s.title)

## Call `weather_node` in isolation

In [ ]:
from agentic_scd.agents.weather import weather_node, assess_weather_signal

state.update(weather_node(state))
print("Output weather_risks:")
for w in state["weather_risks"]:
    print(f" - {w.hub_port}: severity {w.aggregate_severity}/10 over {w.horizon_days}d, peak {w.peak_day}, port risk {w.port_disruption_risk:.0%}, ops {w.affected_operations}")

# Or assess a single signal directly while iterating:
one = assess_weather_signal(state["new_signals"][0])
print("\nsingle:", one.summary)

## Downstream: classify picks up the weather boost

`classify_node` reads `weather_risks` and lifts severity for hubs with real forecast risk, so a severe-weather hub scores higher than keyword matching alone would give.

In [ ]:
from agentic_scd.agents.classify import classify_node

boosted = classify_node(state)["classifications"]
plain = classify_node({"new_signals": state["new_signals"]})["classifications"]
for b, p in zip(boosted, plain):
    print(f" - {b.category}: with weather {b.severity} vs without {p.severity}")

## Iterate here

This is your dev surface: edit the input snapshot (or point `fallback_path` at a live pull), re-run, and watch `weather_node`'s output change. Keep the node signature stable so the rest of the graph is unaffected.